# Lab 1: Numeric Computation in Python — Understanding NumPy Under the Hood

## 🎯 Learning Objectives

By the end of this lab, you will understand:
- How Python's magic methods enable operator overloading
- How NumPy implements numeric operations internally
- The difference between element-wise and matrix operations
- How to build a basic numeric computation library from scratch

**Why This Matters:** Deep learning frameworks like PyTorch and TensorFlow are built on these same principles. Understanding the fundamentals will help you debug issues and build custom operations later.

**Note:** If running in Google Colab, all packages are pre-installed. No setup needed!

## Part 1: Python Magic Methods — The Foundation of Operator Overloading

### What Are Magic Methods?

Magic methods (also called **dunder methods**, short for "double underscore") are special methods that Python calls when you use operators or built-in functions on objects.

### Example: How `+` Really Works

When you write `a + b`, Python translates this to `a.__add__(b)`. Let's see this in action:

In [ ]:
# Regular addition
a = 5
b = 3
print(f"a + b = {a + b}")          # Output: 8
print(f"a.__add__(b) = {a.__add__(b)}")  # Output: 8 (same thing!)

# Even built-in types use magic methods!
print(f"\nType of 5: {type(5)}")
print(f"5 has __add__ method: {hasattr(5, '__add__')}")

## Interactive Visualizations

Below are interactive visualizations to help you understand matrix operations.

**Try clicking on cells to see how operations work!**

In [ ]:
%%html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Matrix Addition — Python</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: sans-serif; background: #fff; color: #1a1a1a; padding: 24px; max-width: 740px; margin: 0 auto; }
  h1 { font-size: 18px; font-weight: 500; margin-bottom: 4px; }
  .subtitle { font-size: 13px; color: #666; margin-bottom: 20px; }
  .bracket { display:inline-flex; align-items:center; font-size:32px; color:#888; line-height:1; }
  .mat { display:grid; gap:6px; padding:10px 12px; }
  .cell {
    width:48px; height:40px; border-radius:6px; border:0.5px solid #ccc;
    display:flex; align-items:center; justify-content:center; font-size:14px; font-weight:500;
    transition:background .2s, border-color .2s, transform .15s;
    background:#fff; color:#1a1a1a;
  }
  .cell.clickable { cursor:pointer; }
  .cell.clickable:hover { transform:scale(1.07); border-color:#888; }
  .cell.hi-a   { background:#E6F1FB; border-color:#185FA5; color:#0C447C; }
  .cell.hi-b   { background:#E1F5EE; border-color:#0F6E56; color:#085041; }
  .cell.hi-out { background:#FAEEDA; border-color:#854F0B; color:#633806; }
  .op-area { min-height:80px; margin-top:20px; padding:16px 20px; border-radius:12px; border:0.5px solid #ddd; background:#f7f7f5; font-size:13px; line-height:1.7; }
  .warn-banner { padding:10px 14px; border-radius:8px; background:#FAEEDA; border:0.5px solid #854F0B; color:#633806; font-size:13px; margin-top:14px; display:none; }
  .code-toggle { margin-top:14px; display:flex; align-items:center; gap:10px; }
  .code-toggle button { padding:5px 14px; border-radius:6px; border:0.5px solid #ccc; background:transparent; color:#1a1a1a; font-size:12px; cursor:pointer; }
  .code-toggle button:hover { background:#f0f0ee; }
  .code-block { margin-top:10px; padding:12px 14px; border-radius:8px; background:#f7f7f5; border:0.5px solid #ddd; font-family:monospace; font-size:12px; line-height:1.8; white-space:pre; overflow-x:auto; display:none; }
  .num-a { color:#185FA5; font-weight:500; }
  .num-b { color:#0F6E56; font-weight:500; }
  .num-o { color:#854F0B; font-weight:500; }
  .sym { color:#888; }
  .hint { font-size:12px; color:#666; margin-top:8px; }
  .label { font-size:12px; color:#666; text-align:center; margin-bottom:6px; font-weight:500; }
  .matrices { display:flex; align-items:center; gap:10px; flex-wrap:wrap; justify-content:center; }
  .key-rule { margin-top:16px; padding:10px 14px; border-radius:8px; border:0.5px solid #ccc; background:#f7f7f5; font-size:12px; color:#444; line-height:1.6; }
  .key-rule b { color:#1a1a1a; }
</style>
</head>
<body>
<h1>Matrix Addition — Pure Python</h1>
<p class="subtitle">Click any cell in the <b>Result</b> matrix to see how it's computed.</p>

<div style="padding:1rem 0">
  <div class="matrices">
    <div>
      <div class="label">Matrix A (3×3)</div>
      <div style="display:flex; align-items:center;">
        <span class="bracket">[</span>
        <div class="mat" style="grid-template-columns:repeat(3,48px)" id="matA"></div>
        <span class="bracket">]</span>
      </div>
    </div>
    <span style="font-size:26px; font-weight:400; color:#888; padding:0 4px">+</span>
    <div>
      <div class="label">Matrix B (3×3)</div>
      <div style="display:flex; align-items:center;">
        <span class="bracket">[</span>
        <div class="mat" style="grid-template-columns:repeat(3,48px)" id="matB"></div>
        <span class="bracket">]</span>
      </div>
    </div>
    <span style="font-size:26px; font-weight:400; color:#888; padding:0 4px">=</span>
    <div>
      <div class="label">Result C (3×3)</div>
      <div style="display:flex; align-items:center;">
        <span class="bracket">[</span>
        <div class="mat" style="grid-template-columns:repeat(3,48px)" id="matC"></div>
        <span class="bracket">]</span>
      </div>
    </div>
  </div>

  <div class="key-rule">
    <b>Key rule:</b> Matrix addition only works when both matrices have the <b>same dimensions</b>. Each element is added to the element at the exact same position — row by row, column by column.
  </div>

  <div class="op-area" id="opArea">
    <span style="color:#888">👆 Click any cell in the <b>Result</b> matrix to see how it's computed</span>
  </div>

  <div class="warn-banner" id="warnBanner">
    ⚠️ <b>Spoiler alert!</b> Try to work out the answer yourself first — then reveal the code to check your thinking.
  </div>

  <div class="code-toggle">
    <button onclick="toggleCode()">🔍 Show Python code</button>
    <span style="font-size:12px; color:#666;">Try the math yourself first!</span>
  </div>
  <div class="code-block" id="codeBlock"># Pure Python matrix addition
def mat_add(A, B):
    rows = len(A)
    cols = len(A[0])
    C = [[0]*cols for _ in range(rows)]
    for i in range(rows):
        for j in range(cols):
            C[i][j] = A[i][j] + B[i][j]
    return C</div>
</div>

<script>
const A = [[1,2,3],[4,5,6],[7,8,9]];
const B = [[9,8,7],[6,5,4],[3,2,1]];
const C = A.map((row,i) => row.map((v,j) => v + B[i][j]));

let codeVisible = false;
let warnShown = false;

function toggleCode(){
  codeVisible = !codeVisible;
  document.getElementById('codeBlock').style.display = codeVisible ? 'block' : 'none';
  document.querySelector('.code-toggle button').textContent = codeVisible ? '🙈 Hide Python code' : '🔍 Show Python code';
  if(codeVisible && !warnShown){
    document.getElementById('warnBanner').style.display = 'block';
    warnShown = true;
  }
  if(!codeVisible) document.getElementById('warnBanner').style.display = 'none';
}

function renderMat(id, data, clickable){
  const el = document.getElementById(id);
  el.innerHTML = '';
  data.forEach((row,i) => row.forEach((v,j) => {
    const d = document.createElement('div');
    d.className = 'cell' + (clickable ? ' clickable' : '');
    d.textContent = v;
    d.dataset.r = i; d.dataset.c = j;
    if(clickable){ d.onclick = () => showStep(i,j); d.title = `Click to compute C[${i}][${j}]`; }
    el.appendChild(d);
  }));
}

function clearHighlight(){
  document.querySelectorAll('.cell').forEach(c => c.classList.remove('hi-a','hi-b','hi-out'));
}

function showStep(ri, ci){
  clearHighlight();

  // Highlight the matching cell in A
  document.querySelectorAll('#matA .cell').forEach(c => {
    if(+c.dataset.r === ri && +c.dataset.c === ci) c.classList.add('hi-a');
  });
  // Highlight the matching cell in B
  document.querySelectorAll('#matB .cell').forEach(c => {
    if(+c.dataset.r === ri && +c.dataset.c === ci) c.classList.add('hi-b');
  });
  // Highlight result cell
  document.querySelectorAll('#matC .cell').forEach(c => {
    if(+c.dataset.r === ri && +c.dataset.c === ci) c.classList.add('hi-out');
  });

  const a = A[ri][ci];
  const b = B[ri][ci];
  const sum = a + b;

  document.getElementById('opArea').innerHTML = `
    <div><b>Computing C[${ri}][${ci}]</b> — adding the element at row ${ri}, column ${ci} from each matrix</div>
    <div style="margin-top:8px;font-size:15px">
      <span class="num-a">A[${ri}][${ci}] = ${a}</span>
      <span class="sym"> + </span>
      <span class="num-b">B[${ri}][${ci}] = ${b}</span>
      <span class="sym"> = </span>
      <span class="num-o">${sum}</span>
    </div>
    <div class="hint">Unlike multiplication, addition is simple: same position in A plus same position in B. No rows or columns to traverse.</div>
    <div class="hint" style="margin-top:4px">Code path: <code>C[${ri}][${ci}] = A[${ri}][${ci}] + B[${ri}][${ci}]</code></div>`;
}

renderMat('matA', A, false);
renderMat('matB', B, false);
renderMat('matC', C, true);
</script>
</body>
</html>


In [ ]:
%%html
<style>
  #matmul-container * { box-sizing: border-box; }
  #matmul-container { font-family: sans-serif; background: #fff; color: #1a1a1a; padding: 24px; max-width: 740px; margin: 0 auto; }
  #matmul-container h1 { font-size: 18px; font-weight: 500; margin-bottom: 4px; }
  #matmul-container .subtitle { font-size: 13px; color: #666; margin-bottom: 20px; }
  #matmul-container .bracket { display:inline-flex; align-items:center; font-size:32px; color:#888; line-height:1; }
  #matmul-container .mat { display:grid; gap:6px; padding:10px 12px; }
  #matmul-container .cell {
    width:48px; height:40px; border-radius:6px; border:0.5px solid #ccc;
    display:flex; align-items:center; justify-content:center; font-size:14px; font-weight:500;
    cursor:pointer; transition:background .2s, border-color .2s, transform .15s;
    background:#fff; color:#1a1a1a;
  }
  #matmul-container .cell:hover { transform:scale(1.07); border-color:#888; }
  #matmul-container .cell.hi-row  { background:#E6F1FB; border-color:#185FA5; color:#0C447C; }
  #matmul-container .cell.hi-col  { background:#E1F5EE; border-color:#0F6E56; color:#085041; }
  #matmul-container .cell.hi-out  { background:#FAEEDA; border-color:#854F0B; color:#633806; }
  #matmul-container .op-area { min-height:80px; margin-top:20px; padding:16px 20px; border-radius:12px; border:0.5px solid #ddd; background:#f7f7f5; font-size:13px; line-height:1.7; }
  #matmul-container .warn-banner { padding:10px 14px; border-radius:8px; background:#FAEEDA; border:0.5px solid #854F0B; color:#633806; font-size:13px; margin-top:14px; display:none; }
  #matmul-container .code-toggle { margin-top:14px; display:flex; align-items:center; gap:10px; }
  #matmul-container .code-toggle button { padding:5px 14px; border-radius:6px; border:0.5px solid #ccc; background:transparent; color:#1a1a1a; font-size:12px; cursor:pointer; }
  #matmul-container .code-toggle button:hover { background:#f0f0ee; }
  #matmul-container .code-block { margin-top:10px; padding:12px 14px; border-radius:8px; background:#f7f7f5; border:0.5px solid #ddd; font-family:monospace; font-size:12px; line-height:1.8; white-space:pre; overflow-x:auto; display:none; }
  #matmul-container .num-r { color:#185FA5; font-weight:500; }
  #matmul-container .num-g { color:#0F6E56; font-weight:500; }
  #matmul-container .num-o { color:#854F0B; font-weight:500; }
  #matmul-container .sym { color:#888; }
  #matmul-container .hint { font-size:12px; color:#666; margin-top:8px; }
  #matmul-container .label { font-size:12px; color:#666; text-align:center; margin-bottom:6px; font-weight:500; }
  #matmul-container .matrices { display:flex; align-items:center; gap:10px; flex-wrap:wrap; justify-content:center; }
</style>

<div id="matmul-container">
  <h1>Matrix Multiplication — Pure Python</h1>
  <p class="subtitle">Click any cell in the <b>Result</b> matrix to see how it's computed as a dot product.</p>

  <div style="padding:1rem 0">
    <div class="matrices">
      <div>
        <div class="label">Matrix A (2×3)</div>
        <div style="display:flex; align-items:center;">
          <span class="bracket">[</span>
          <div class="mat" style="grid-template-columns:repeat(3,48px)" id="matmul-matA"></div>
          <span class="bracket">]</span>
        </div>
      </div>
      <span style="font-size:26px; font-weight:400; color:#888; padding:0 4px">×</span>
      <div>
        <div class="label">Matrix B (3×2)</div>
        <div style="display:flex; align-items:center;">
          <span class="bracket">[</span>
          <div class="mat" style="grid-template-columns:repeat(2,48px)" id="matmul-matB"></div>
          <span class="bracket">]</span>
        </div>
      </div>
      <span style="font-size:26px; font-weight:400; color:#888; padding:0 4px">=</span>
      <div>
        <div class="label">Result C (2×2)</div>
        <div style="display:flex; align-items:center;">
          <span class="bracket">[</span>
          <div class="mat" style="grid-template-columns:repeat(2,48px)" id="matmul-matC"></div>
          <span class="bracket">]</span>
        </div>
      </div>
    </div>

    <div class="op-area" id="matmul-opArea">
      <span style="color:#888">👆 Click any cell in the <b>Result</b> matrix to see how it's computed</span>
    </div>

    <div class="warn-banner" id="matmul-warnBanner">
      ⚠️ <b>Spoiler alert!</b> Try to work out the answer yourself first — then reveal the code to check your thinking.
    </div>

    <div class="code-toggle">
      <button onclick="matmulToggleCode()">🔍 Show Python code</button>
      <span style="font-size:12px; color:#666;">Try the math yourself first!</span>
    </div>
    <div class="code-block" id="matmul-codeBlock"># Pure Python matrix multiplication
def mat_mul(A, B):
    rows_A, cols_A = len(A), len(A[0])
    cols_B = len(B[0])
    C = [[0]*cols_B for _ in range(rows_A)]
    for i in range(rows_A):
        for j in range(cols_B):
            for k in range(cols_A):
                C[i][j] += A[i][k] * B[k][j]
    return C</div>
  </div>
</div>

<script>
(function() {
  const A = [[1,2,3],[4,5,6]];
  const B = [[7,8],[9,10],[11,12]];
  const C = A.map((rowA,i)=>B[0].map((_,j)=>A[i].reduce((s,_,k)=>s+A[i][k]*B[k][j],0)));

  let codeVisible = false;
  let warnShown = false;

  window.matmulToggleCode = function() {
    codeVisible = !codeVisible;
    document.getElementById('matmul-codeBlock').style.display = codeVisible ? 'block' : 'none';
    document.querySelector('#matmul-container .code-toggle button').textContent = codeVisible ? '🙈 Hide Python code' : '🔍 Show Python code';
    if(codeVisible && !warnShown){
      document.getElementById('matmul-warnBanner').style.display = 'block';
      warnShown = true;
    }
    if(!codeVisible) document.getElementById('matmul-warnBanner').style.display = 'none';
  };

  function renderMat(id, data, clickable){
    const el=document.getElementById(id);
    if (!el) return;
    el.innerHTML='';
    data.forEach((row,i)=>row.forEach((v,j)=>{
      const d=document.createElement('div');
      d.className='cell'; d.textContent=v;
      d.dataset.r=i; d.dataset.c=j;
      if(clickable){ d.onclick=()=>showStep(i,j); d.title=`Click to compute C[${i}][${j}]`; }
      el.appendChild(d);
    }));
  }

  function clearHighlight(){
    document.querySelectorAll('#matmul-container .cell').forEach(c=>c.classList.remove('hi-row','hi-col','hi-out'));
  }

  function showStep(ri,ci){
    clearHighlight();
    document.querySelectorAll('#matmul-matA .cell').forEach(c=>{if(+c.dataset.r===ri)c.classList.add('hi-row');});
    document.querySelectorAll('#matmul-matB .cell').forEach(c=>{if(+c.dataset.c===ci)c.classList.add('hi-col');});
    document.querySelectorAll('#matmul-matC .cell').forEach(c=>{if(+c.dataset.r===ri&&+c.dataset.c===ci)c.classList.add('hi-out');});

    const rowA=A[ri]; const colB=B.map(r=>r[ci]);
    const terms=rowA.map((v,k)=>`<span class="num-r">${v}</span><span class="sym">×</span><span class="num-g">${colB[k]}</span>`).join('<span class="sym"> + </span>');
    const sum=rowA.reduce((s,v,k)=>s+v*colB[k],0);
    const termNums=rowA.map((v,k)=>(v*colB[k])).join(' + ');

    const opArea = document.getElementById('matmul-opArea');
    if (opArea) {
      opArea.innerHTML=`
        <div><b>Computing C[${ri}][${ci}]</b> — dot product of <span style="color:#185FA5">row ${ri} of A</span> · <span style="color:#0F6E56">col ${ci} of B</span></div>
        <div style="margin-top:8px;font-size:15px">${terms} <span class="sym">=</span> <span style="color:#c67700">${termNums}</span> <span class="sym">=</span> <span class="num-o">${sum}</span></div>
        <div class="hint">Each pair multiplies element-by-element along the row/column, then sums.</div>`;
    }
  }

  renderMat('matmul-matA',A,false);
  renderMat('matmul-matB',B,false);
  renderMat('matmul-matC',C,true);
})();
</script>


### Why This Matters for Deep Learning

When you write `c = a + b` in NumPy or PyTorch:
```python
import numpy as np
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
c = a + b  # This calls a.__add__(b)
```

The library implements custom `__add__` that:
1. Performs element-wise addition
2. Validates shapes match
3. Uses optimized C code for speed

Later in this course, we'll use the same technique to build a **computation graph** that tracks operations for automatic differentiation!

### Key Magic Methods for Numeric Computation

| Magic Method | Operator | Called When | Example |
|--------------|----------|-------------|----------|
| `__add__(self, other)` | `+` | Addition | `a + b` |
| `__sub__(self, other)` | `-` | Subtraction | `a - b` |
| `__mul__(self, other)` | `*` | Multiplication | `a * b` |
| `__truediv__(self, other)` | `/` | Division | `a / b` |
| `__matmul__(self, other)` | `@` | Matrix multiply | `a @ b` |
| `__repr__(self)` | `print()` | String representation | `print(a)` |
| `__getitem__(self, key)` | `[]` | Indexing | `a[0]` |

**Note:** There are also "reverse" versions (`__radd__`, `__rmul__`, etc.) for when the left operand doesn't support the operation.

## Part 2: Building a Simple Numeric Array Class

Let's build our own `Array` class to understand how NumPy works internally.

### Design Goals:
1. Store data in a Python list
2. Support element-wise operations (+, -, *, /)
3. Validate shapes match for operations
4. Pretty printing

### Demo: A Complete Implementation

Study this implementation carefully — you'll build something similar in the exercise!

In [ ]:
class SimpleArray:
    """A simple 1D array class demonstrating operator overloading."""
    
    def __init__(self, data):
        """Initialize with a Python list."""
        self.data = list(data)  # Store as list
        self.size = len(data)   # Track size
    
    def __add__(self, other):
        """Element-wise addition: self + other"""
        # Validate sizes match
        if self.size != other.size:
            raise ValueError(f"Size mismatch: {self.size} vs {other.size}")
        
        # Perform element-wise addition
        result = [self.data[i] + other.data[i] for i in range(self.size)]
        return SimpleArray(result)
    
    def __mul__(self, other):
        """Element-wise multiplication: self * other"""
        if self.size != other.size:
            raise ValueError(f"Size mismatch: {self.size} vs {other.size}")
        
        result = [self.data[i] * other.data[i] for i in range(self.size)]
        return SimpleArray(result)
    
    def __repr__(self):
        """Pretty printing."""
        return f"SimpleArray({self.data})"

# Test it!
a = SimpleArray([1, 2, 3])
b = SimpleArray([4, 5, 6])

print(f"a = {a}")
print(f"b = {b}")
print(f"a + b = {a + b}")  # Calls a.__add__(b)
print(f"a * b = {a * b}")  # Calls a.__mul__(b)

### How This Compares to NumPy

Let's compare our implementation with NumPy:

In [ ]:
import numpy as np

# NumPy arrays
a_np = np.array([1, 2, 3])
b_np = np.array([4, 5, 6])

print("NumPy:")
print(f"a_np + b_np = {a_np + b_np}")  # Same result!
print(f"a_np * b_np = {a_np * b_np}")  # Same result!

print("\nKey Differences:")
print(f"1. NumPy uses C arrays (fast): {type(a_np.data)}")
print(f"2. Our class uses Python lists (slow): {type(a.data)}")
print(f"3. NumPy supports broadcasting (flexible shapes)")
print(f"4. NumPy has hundreds of optimized operations")

## Part 3: Matrix Multiplication — The Most Important Operation

### Why Matrix Multiplication Matters

Matrix multiplication is the **core operation** in deep learning:
- Forward pass: `output = input @ weights`
- Backward pass: Compute gradients via matrix products

Understanding it deeply is crucial!

### Element-wise vs Matrix Multiplication

**Element-wise multiplication** (what we did above with `*`):
```
[1, 2, 3] * [4, 5, 6] = [1*4, 2*5, 3*6] = [4, 10, 18]
```

**Matrix multiplication** (what `@` does in NumPy):
```
A @ B where A is shape (m, n) and B is shape (n, p)
Result is shape (m, p)
```

### How Matrix Multiplication Works

For matrices A (2×3) and B (3×2):

```
A = [[1, 2, 3],      B = [[7, 8],
     [4, 5, 6]]           [9, 10],
                          [11, 12]]

C = A @ B  (result will be 2×2)

C[0,0] = A[0,:] · B[:,0] = 1*7 + 2*9 + 3*11 = 7 + 18 + 33 = 58
C[0,1] = A[0,:] · B[:,1] = 1*8 + 2*10 + 3*12 = 8 + 20 + 36 = 64
C[1,0] = A[1,:] · B[:,0] = 4*7 + 5*9 + 6*11 = 28 + 45 + 66 = 139
C[1,1] = A[1,:] · B[:,1] = 4*8 + 5*10 + 6*12 = 32 + 50 + 72 = 154

C = [[58, 64],
     [139, 154]]
```

**Key Rules:**
1. Inner dimensions must match: `(m, n) @ (n, p)` ✓
2. Result shape: `(m, p)`
3. Each element is a dot product of a row and column

In [ ]:
# Demo: Matrix multiplication step-by-step
import numpy as np

A = np.array([[1, 2, 3],
              [4, 5, 6]])

B = np.array([[7, 8],
              [9, 10],
              [11, 12]])

print("Matrix A (2×3):")
print(A)
print("\nMatrix B (3×2):")
print(B)

# NumPy matrix multiplication
C = A @ B
print("\nResult C = A @ B (2×2):")
print(C)

# Manual verification of first element
print("\nManual calculation of C[0,0]:")
print(f"A[0,:] = {A[0,:]}")
print(f"B[:,0] = {B[:,0]}")
print(f"Dot product = {A[0,0]*B[0,0]} + {A[0,1]*B[1,0]} + {A[0,2]*B[2,0]} = {A[0,0]*B[0,0] + A[0,1]*B[1,0] + A[0,2]*B[2,0]}")
print(f"Matches C[0,0] = {C[0,0]} ✓")

### Visualizing the Process

Think of matrix multiplication as:
1. Take a **row** from the left matrix
2. Take a **column** from the right matrix
3. Multiply corresponding elements
4. Sum them up
5. That's one element in the result!
6. Repeat for all row-column combinations

```
      [col0  col1]
        ↓     ↓
[row0]  •     •    ← Each • is a dot product
[row1]  •     •
```

## Exercise: Build Your Own Array Class

### Goal

Implement an `Array` class that supports:
1. Element-wise addition (`+`)
2. Element-wise subtraction (`-`)
3. Element-wise multiplication (`*`)
4. Matrix multiplication (`@`)
5. Pretty printing

### Requirements

- Store data as a 2D list: `[[row0], [row1], ...]`
- Track shape: `(rows, cols)`
- Validate shapes for operations
- **NO NUMPY ALLOWED** in your implementation (you need to understand the internals first!)

### Starter Code

Fill in the TODOs below:

In [ ]:
class Array:
    """A 2D array class for numeric computation (no NumPy allowed!)."""
    
    def __init__(self, data):
        """
        Initialize array from 2D list.
        
        Args:
            data: 2D list like [[1, 2], [3, 4]]
        """
        # TODO: Store data as 2D list
        # TODO: Calculate self.rows = number of rows
        # TODO: Calculate self.cols = number of columns (length of first row)
        # TODO: Store self.shape = (rows, cols)
        raise NotImplementedError("Implement __init__")
    
    def __add__(self, other):
        """
        Element-wise addition.
        
        Example: [[1, 2], [3, 4]] + [[5, 6], [7, 8]] = [[6, 8], [10, 12]]
        """
        # TODO: Check shapes match: self.shape == other.shape
        # TODO: Create result list with same shape
        # TODO: For each element: result[i][j] = self.data[i][j] + other.data[i][j]
        # TODO: Return new Array(result)
        raise NotImplementedError("Implement __add__")
    
    def __sub__(self, other):
        """
        Element-wise subtraction.
        
        Example: [[5, 6], [7, 8]] - [[1, 2], [3, 4]] = [[4, 4], [4, 4]]
        """
        # TODO: Similar to __add__, but subtract instead
        raise NotImplementedError("Implement __sub__")
    
    def __mul__(self, other):
        """
        Element-wise multiplication (NOT matrix multiplication!).
        
        Example: [[1, 2], [3, 4]] * [[2, 2], [2, 2]] = [[2, 4], [6, 8]]
        """
        # TODO: Similar to __add__, but multiply instead
        raise NotImplementedError("Implement __mul__")
    
    def __matmul__(self, other):
        """
        Matrix multiplication: self @ other
        
        Rules:
        - self.cols must equal other.rows
        - Result shape: (self.rows, other.cols)
        - result[i][j] = sum(self[i][k] * other[k][j] for k in range(self.cols))
        
        Example:
        [[1, 2],  @ [[5, 6],  = [[19, 22],
         [3, 4]]     [7, 8]]     [43, 50]]
        
        Calculation:
        result[0][0] = 1*5 + 2*7 = 19
        result[0][1] = 1*6 + 2*8 = 22
        result[1][0] = 3*5 + 4*7 = 43
        result[1][1] = 3*6 + 4*8 = 50
        """
        # TODO: Check self.cols == other.rows (inner dimensions must match)
        # TODO: Create result with shape (self.rows, other.cols)
        # TODO: For each i, j:
        #       result[i][j] = sum of self[i][k] * other[k][j] for all k
        # TODO: Return new Array(result)
        raise NotImplementedError("Implement __matmul__")
    
    def __repr__(self):
        """
        String representation for printing.
        
        Should print like:
        Array([[1, 2],
               [3, 4]])
        """
        # TODO: Format data nicely
        # Hint: Use '\n       '.join() for multi-line formatting
        raise NotImplementedError("Implement __repr__")

# Don't modify the tests - they should pass when your implementation is correct

### Test Your Implementation

Run these tests to validate your Array class:

In [ ]:
print("=" * 60)
print("Test 1: Element-wise Addition")
print("=" * 60)

A = Array([[1, 2], [3, 4]])
B = Array([[5, 6], [7, 8]])
C = A + B

print(f"A = {A}")
print(f"B = {B}")
print(f"C = A + B = {C}")

expected = [[6, 8], [10, 12]]
assert C.data == expected, f"Expected {expected}, got {C.data}"
print("✓ Addition: PASS\n")

In [ ]:
print("=" * 60)
print("Test 2: Element-wise Subtraction")
print("=" * 60)

D = B - A
print(f"D = B - A = {D}")

expected = [[4, 4], [4, 4]]
assert D.data == expected, f"Expected {expected}, got {D.data}"
print("✓ Subtraction: PASS\n")

In [ ]:
print("=" * 60)
print("Test 3: Element-wise Multiplication")
print("=" * 60)

E = Array([[2, 2], [2, 2]])
F = A * E
print(f"A = {A}")
print(f"E = {E}")
print(f"F = A * E = {F}")

expected = [[2, 4], [6, 8]]
assert F.data == expected, f"Expected {expected}, got {F.data}"
print("✓ Element-wise multiplication: PASS\n")

In [ ]:
print("=" * 60)
print("Test 4: Matrix Multiplication")
print("=" * 60)

# 2x2 @ 2x2 = 2x2
G = A @ B
print(f"A = {A}")
print(f"B = {B}")
print(f"G = A @ B = {G}")

# Manual calculation:
# G[0][0] = 1*5 + 2*7 = 5 + 14 = 19
# G[0][1] = 1*6 + 2*8 = 6 + 16 = 22
# G[1][0] = 3*5 + 4*7 = 15 + 28 = 43
# G[1][1] = 3*6 + 4*8 = 18 + 32 = 50
expected = [[19, 22], [43, 50]]
assert G.data == expected, f"Expected {expected}, got {G.data}"
print("✓ Matrix multiplication: PASS\n")

In [ ]:
print("=" * 60)
print("Test 5: Non-square Matrix Multiplication")
print("=" * 60)

# 2x3 @ 3x2 = 2x2
H = Array([[1, 2, 3], [4, 5, 6]])
I = Array([[7, 8], [9, 10], [11, 12]])
J = H @ I

print(f"H (2x3) = {H}")
print(f"I (3x2) = {I}")
print(f"J = H @ I (2x2) = {J}")

# Manual: 
# J[0][0] = 1*7 + 2*9 + 3*11 = 7 + 18 + 33 = 58
# J[0][1] = 1*8 + 2*10 + 3*12 = 8 + 20 + 36 = 64
# J[1][0] = 4*7 + 5*9 + 6*11 = 28 + 45 + 66 = 139
# J[1][1] = 4*8 + 5*10 + 6*12 = 32 + 50 + 72 = 154
expected = [[58, 64], [139, 154]]
assert J.data == expected, f"Expected {expected}, got {J.data}"
print("✓ Non-square matmul: PASS\n")

In [ ]:
print("=" * 60)
print("🎉 All Tests Passed!")
print("=" * 60)
print("\nYou've successfully implemented a basic numeric computation library!")
print("This is essentially how NumPy works under the hood (but much faster).")

## Bonus Challenge: Compare with NumPy

Now that you understand the internals, let's compare your implementation with NumPy:

In [ ]:
import numpy as np

# Your implementation
a_custom = Array([[1, 2, 3], [4, 5, 6]])
b_custom = Array([[7, 8], [9, 10], [11, 12]])
c_custom = a_custom @ b_custom

# NumPy implementation
a_np = np.array([[1, 2, 3], [4, 5, 6]])
b_np = np.array([[7, 8], [9, 10], [11, 12]])
c_np = a_np @ b_np

print("Your implementation:")
print(c_custom)
print("\nNumPy:")
print(c_np)
print("\n✓ Results match!" if c_custom.data == c_np.tolist() else "✗ Results don't match")

print("\n" + "="*60)
print("Key Differences:")
print("="*60)
print("1. Speed: NumPy uses optimized C code (100-1000x faster)")
print("2. Memory: NumPy uses contiguous memory (cache-friendly)")
print("3. Features: NumPy has broadcasting, fancy indexing, etc.")
print("4. Types: NumPy supports many numeric types (float32, int64, etc.)")
print("\nBut the LOGIC is the same! You now understand what's happening under the hood.")

## Summary

### What You've Learned

✅ **Magic methods** enable operator overloading in Python

✅ **Element-wise operations** work on corresponding elements

✅ **Matrix multiplication** computes dot products of rows and columns

✅ **NumPy internals** — you built a mini-version from scratch!

### Why This Matters

In the next labs, you'll:
- Build a `Value` class that tracks operations (computation graph)
- Use **NumPy** for efficient numeric computation (you've earned it!)
- Implement automatic differentiation for neural networks

Understanding these fundamentals will help you:
- Debug deep learning code more effectively
- Build custom operations when needed
- Understand how PyTorch and TensorFlow work internally

### Next Steps

In **Lab 2**, we'll introduce:
- Computation graphs for tracking operations
- The `Value` class that records its history
- NumPy for efficient numeric computation (finally!)
- Forward propagation through networks

**Great work!** 🎉 You've built the foundation for understanding deep learning frameworks.